# Motion Detection

### What Is Motion Detection?

Optical flow asked:
- where did each pixel move?

Motion detection asks something simpler:
- did anything change between this frame and the last?

No neural network, no tracking, no classes:
- just raw pixel arithmetic
- extremely fast
- foundation of most security camera systems

Real use case:
- camera watches 24/7
- runs YOLO only when motion is detected
- saves massive compute

### Core Idea — Frame Differencing

Two frames as numpy arrays:

```
Frame 1:  [[120, 118, 115, ...], ...]
Frame 2:  [[120, 118, 200, ...], ...]  ← something moved here
```

Subtract them:

```
No change:  120 - 120 = 0    → black pixel
Change:     200 - 115 = 85   → bright pixel
```

Result:
- black everywhere nothing moved
- bright wherever something changed

### Full Pipeline

```
frame1, frame2
↓
absdiff → bright where changed
↓
threshold → ignore small noise
↓
dilate → fill gaps, merge nearby blobs
↓
findContours → outline each region
↓
draw bounding boxes around motion regions
```

### Why Each Step?

**absdiff instead of plain subtraction:**
- subtraction can go negative
- we want magnitude of change regardless of direction
- `absdiff` always returns positive values

**threshold:**
- camera noise and lighting flicker cause tiny pixel differences
- `diff = 2` is noise, not motion
- only keep differences above a minimum value

**dilation:**
- thresholding leaves scattered disconnected blobs
- dilation expands each white pixel outward
- nearby blobs merge into one solid region

```
Before dilation:    After dilation:
. . ░ . . ░ . .    . ░ ░ ░ ░ ░ ░ .
. ░ . . . . ░ .    ░ ░ ░ ░ ░ ░ ░ ░
. . ░ . . ░ . .    . ░ ░ ░ ░ ░ ░ .
```

**findContours:**
- finds outlines of white regions in binary image
- each contour = one motion region
- we draw a bounding box around each one

### Setup

In [1]:
import cv2
import numpy as np

### Constants

In [ ]:
DIFF_THRESHOLD  = 25     # minimum pixel difference to count as motion
DILATE_KERNEL   = 21     # dilation kernel size — larger = more gap filling
MIN_AREA        = 500    # minimum contour area to draw a box

# Cool display:
# DIFF_THRESHOLD = 10
# DILATE_KERNEL = 1
# MIN_AREA = 1000

| Constant | Meaning |
| -------- | ------- |
| `DIFF_THRESHOLD` | pixel difference below this = noise, above = motion |
| `DILATE_KERNEL` | expansion size — larger merges more blobs together |
| `MIN_AREA` | filters out tiny noise blobs that survived dilation |

### Motion Detection Function

In [3]:
def detect_motion(prev_gray, gray):

    # Difference
    diff = cv2.absdiff(prev_gray, gray)

    # Threshold
    _, thresh = cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)

    # Dilate
    kernel = np.ones((DILATE_KERNEL, DILATE_KERNEL), np.uint8)
    dilated = cv2.dilate(thresh, kernel)

    # Find contours
    contours, hierarchy = cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    boxes = []

    for contour in contours:
        if cv2.contourArea(contour) < MIN_AREA:
            continue
        x, y, w, h = cv2.boundingRect(contour)
        boxes.append((x, y, x + w, y + h))

    return boxes, dilated

#### `cv2.absdiff(prev_gray, gray)`
- `abs` = Absolute, `diff` = Difference
- absolute difference between two grayscale frames
- returns same size image with difference magnitudes

| Attribute | Meaning |
| --------- | ------- |
| `prev_gray` | previous grayscale frame |
| `gray` | current grayscale frame |

#### `cv2.threshold(diff, DIFF_THRESHOLD, 255, cv2.THRESH_BINARY)`
- `THRESH` = Threshold
- converts difference image to binary — motion or not
- returns `retval` (threshold used) and `thresh` (result image)
- `retval` discarded with `_` — we already know the threshold we set

| Attribute | Meaning |
| --------- | ------- |
| `diff` | input difference image |
| `DIFF_THRESHOLD` | pixels above this → white, below → black |
| `255` | value to assign to pixels above threshold |
| `THRESH_BINARY` | simple on/off threshold — no gradients |

#### `cv2.dilate(thresh, kernel)`
- expands white regions outward by kernel size
- nearby blobs merge into one solid region

| Attribute | Meaning |
| --------- | ------- |
| `thresh` | input binary image |
| `kernel` | shape and size of expansion — `np.ones((21,21))` expands equally in all directions |

#### `cv2.findContours(dilated, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)`
- finds outlines of white regions in a binary image
- `RETR` = Retrieval mode — how contours are organised
- `CHAIN_APPROX` = Chain Approximation — how contour points are stored
- returns `contours` (list of outlines) and `hierarchy` (parent-child relationships, not needed here)

| Attribute | Meaning |
| --------- | ------- |
| `dilated` | input binary image to find contours in |
| `RETR_EXTERNAL` | outermost contours only, ignores holes inside regions |
| `RETR_LIST` | all contours, no hierarchy |
| `RETR_TREE` | all contours with full parent-child hierarchy |
| `CHAIN_APPROX_SIMPLE` | stores only corner points, compresses straight lines |
| `CHAIN_APPROX_NONE` | stores every single point along the contour |

#### `cv2.contourArea(contour)`
- returns area of contour in pixels
- used to filter out tiny noise blobs that survived dilation

#### `cv2.boundingRect(contour)`
- returns `x, y, w, h` of the bounding rectangle around the contour
- `x, y` = top-left corner
- `w, h` = width and height
- we convert to `x1, y1, x2, y2` format to stay consistent with other notebooks

### Webcam Loop

In [ ]:
cap = cv2.VideoCapture(0)

ret, prev_frame = cap.read()
prev_gray = cv2.cvtColor(prev_frame, cv2.COLOR_BGR2GRAY)

while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Detect
    boxes, dilated = detect_motion(prev_gray, gray)

    # Draw
    motion_detected = len(boxes) > 0    # Flag

    for (x1, y1, x2, y2) in boxes:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

    status_text  = "Motion Detected" if motion_detected else "No Motion"
    status_color = (0, 255, 0) if motion_detected else (0, 0, 255)

    cv2.putText(frame, status_text, (20, 40),
                cv2.FONT_HERSHEY_SIMPLEX, 1, status_color, 2)

    # Show both main frame and dilated mask
    cv2.imshow("Motion Detection", frame)
    cv2.imshow("Dilated Mask", dilated)

    prev_gray = gray

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

#### Two windows
- `Motion Detection` — main frame with bounding boxes
- `Dilated Mask` — binary mask after dilation, useful for tuning constants

Tuning guide:
- too much noise → increase `DIFF_THRESHOLD`
- blobs not merging → increase `DILATE_KERNEL`
- small noise boxes appearing → increase `MIN_AREA`

#### `prev_gray = gray`
- update previous frame at end of every loop
- always comparing to the immediately previous frame

### Summary

| Concept | What It Means |
| ------- | ------------- |
| Frame differencing | subtract two frames to find changed pixels |
| `absdiff` | Absolute Difference — always positive result |
| `threshold` | converts difference image to binary — motion or not |
| `THRESH_BINARY` | simple on/off — above threshold = white, below = black |
| Dilation | expands white pixels outward, merges nearby blobs |
| `findContours` | finds outlines of white regions in binary image |
| `RETR_EXTERNAL` | Retrieval External — outermost contours only |
| `CHAIN_APPROX_SIMPLE` | Chain Approximation Simple — corner points only |
| `contourArea` | area of contour in pixels — filters noise blobs |
| `boundingRect` | bounding box around a contour as `x, y, w, h` |

Next up — action recognition. Instead of detecting what objects are present or where pixels moved, we classify what action is being performed across multiple frames.